# Reproducing Tingley and Buzsaki (2018) on DANDI:000213

Paper. Tingley D, Buzsaki G (2018). Transformation of a Spatial Map across the Hippocampal-Lateral Septal Circuit. Neuron 98, 1229-1242. doi:10.1016/j.neuron.2018.04.028

Dataset. DANDI:000213, version 0.261109.2219 (doi:10.48324/dandi.000213/0.261109.2219), streamed from the archive. The paper cites no version. 0.261109.2219 is the most recent published version (2026-11-09). Its 67 files have the same paths as 0.250624.0438, but each is smaller (1,182 GB in all against 1,527 GB); the cell below prints this. The file descriptions say the LFP was taken from the earlier version and stored as the lab's int16 values instead of float32. The claims file's description (units with region labels, trials with linearized positions, LFP) matches it, so it is pinned.

Claims. `claims/tingley-2018.yaml`, written from the NIH author manuscript (PMC6605060) with its supplement, and checked by a second reader. This attempt is blind: no earlier attempt on the dataset was seen.

Method in brief. Each session's spikes within its trials are mapped to the lab's linearized position and to the CA1 theta phase (6 to 9 Hz, Hilbert). Per unit and included condition (>= 10 trials, >= 1.5 spikes per trial), the single-cell maximum correlation decoder runs on 20-bin smoothed rate and phase maps over 10 random 60/40 splits. Readings of the decoder: split by whole trials (a) or by trial-by-bin samples (b); 201 template bins (a) or 1.5 cm bins (b). Where no spike falls within a bin's smoothing window the phase is undefined; the paper does not say how such bins are treated. They are left out (adopted assumption), and setting them to phase 0 is shown as the alternative. Reading codes: aa0, ba0, ab0, bb0 (split, bins; left out) and aa1, ba1 (set to 0).

| claim | tested | outcome |
|---|---|---|
| ls-neuron-count | yes | reproduced |
| hpc-unit-count | yes | not reproduced |
| ls-active-neurons | yes | reproduced |
| fig2-ls-phase-vs-shuffle | yes | reproduced (assumed) |
| fig2-ls-phase-beats-rate | yes | partially reproduced (assumed) |
| fig2e-ls-phase-lower-mse | yes | reproduced (assumed) |
| fig2fg-mse-difference | yes | not reproduced (assumed) |
| fig1i-hpc-rate-phase-equivalent | yes | not reproduced (assumed) |
| ls-lower-peak-info | yes | partially reproduced |
| ls-few-place-fields | yes | partially reproduced |
| hpc-place-field-properties | yes | partially reproduced |
| ls-place-fields-unstable | yes | partially reproduced |
| fig1e-phase-info-smaller-scale | yes | partially reproduced (assumed) |
| fig1g-phase-more-info-fine-scale | yes | partially reproduced (assumed) |
| fig3-ls-phase-reliable | yes | reproduced |
| fig4d-depth-gradient | yes | partially reproduced (assumed) |
| fig4e-onset-depth | yes | not reproduced (assumed) |
| fig4e-onset-mediolateral | yes | reproduced (assumed) |
| speed-independent | yes | partially reproduced |
| fig5-assemblies | no | blocked, too_large |
| fig5g-assembly-neurons-stronger-phase | no | blocked, too_large (depends on fig5-assemblies) |
| figs5-assembly-independent-of-locking | no | blocked, too_large (depends on fig5-assemblies) |
| fig6-session-counts | yes | partially reproduced |
| fig6c-ca3-coherence-increases | yes | reproduced |
| fig6d-cfc-stronger-ca3 | yes | reproduced |
| fig6e-assembly-position | no | blocked, too_large (depends on fig5-assemblies) |
| figs7-ensemble-ls-phase-best | yes | not reproduced (assumed) |
| figs2-rate-phase-independent | yes | not reproduced |
| figs3-ls-major-target | no | blocked, other (Allen data not analysed in this run) |
| figs6-reference-frame | no | blocked, metadata_insufficient |
| figs1f-trial-counts | yes | partially reproduced |
| fig7-dynamic-weighting-model | no | blocked, other (model not run in this run) |


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, warnings, resource
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tingley as tg
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print("DANDI:%s version %s" % (tg.DANDISET, tg.VERSION))

In [ ]:
# Assets of the pinned version, and how they differ from the previous published version
assets = tg.list_assets()
prev = tg.list_assets("0.250624.0438")
pp = {a["path"]: a["size"] for a in prev}
same_paths = sum(a["path"] in pp for a in assets)
print(f"{len(assets)} assets in {tg.VERSION}, {len(prev)} in 0.250624.0438, {same_paths} paths in both")
print(f"total size {sum(a['size'] for a in assets)/1e9:.0f} GB against {sum(pp.values())/1e9:.0f} GB")
smaller = [a["path"] for a in assets if a["path"] in pp and a["size"] < pp[a["path"]]]
print(f"{len(smaller)} files are smaller in {tg.VERSION} than in 0.250624.0438")

In [ ]:
# Stream and reduce every session (4 at a time); then analyse in 4 processes
SUBSET = None
todo = assets if SUBSET is None else [a for a in assets if any(s in a["path"] for s in SUBSET)]
t = time.time()
with ThreadPoolExecutor(4) as ex:
    sessions = list(ex.map(lambda a: tg.load_session(a, lfp_measures=tg.fig6_lfp_measures), todo))
print(f"loaded {len(sessions)} sessions in {time.time()-t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")
seeds = {S["session"]: 1000 + i for i, S in enumerate(sessions)}
ctx = mp.get_context("fork")
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    parts = list(ex.map(tg.analyse_session, sessions, [seeds[S["session"]] for S in sessions]))
D = pd.DataFrame([r for p in parts for r in p])
print(f"decoders, place fields and information for {len(D)} unit-conditions in {time.time()-t:.0f} s")

In [ ]:
# What was loaded: subjects, sessions, units by region, trials by task, and checks of the description
U_rows = []
for S in sessions:
    T = S["trials"]
    nt = len(T["start"])
    for ui, U in enumerate(S["units"] or []):
        U_rows.append({"session": S["session"], "subject": S["subject"], "unit": ui, "region": U["region"],
                       "n_in_trials": len(U["t"]), "ntrials": nt, "site": U["site"],
                       "n_incl": len(tg.included_conditions(S, U))})
UN = pd.DataFrame(U_rows)
print("sessions per subject:", dict(Counter(S["subject"] for S in sessions)))
print("sessions without a units table:", [S["session"] for S in sessions if S["units"] is None])
print("units by subject and region:")
print(pd.crosstab(UN.subject, UN.region, margins=True))
def task_of(S):
    T = S["trials"]
    tt = T["trial_type"]
    if (tt == "").all():
        task = S["description"].split("Task: ")[1].split(" (")[0]
        return np.array([task] * len(tt))
    return tt
tasks = Counter(x for S in sessions for x in task_of(S))
print("trials by task:", dict(tasks))
print("sessions whose description mentions a rotation:", [S["session"] for S in sessions if "rotat" in S["description"].lower()])
print("condition types:", dict(Counter(x for S in sessions for x in S["template_type"].values())))
print("trial counts per session: min %d, median %d, max %d" % tuple(np.percentile([len(S["trials"]["start"]) for S in sessions], [0, 50, 100])))
print("position units:", dict(Counter(S["pos_unit"] for S in sessions)), "sampling rates (Hz):", sorted(set(round(S["pos_rate"]) for S in sessions)))
scales = Counter(tg.length_scale(S) for S in sessions)
print("metres per stored unit, from template lengths:", dict(scales))
for S in sessions:
    L = np.median(list(S["template_len"].values()))
    if S["pos_unit"] == "meters" and tg.length_scale(S) == 1e-3:
        pass
mm = [S["session"] for S in sessions if tg.length_scale(S) == 1e-3]
print(f"{len(mm)} sessions labelled 'meters' whose template paths are {min(np.median(list(S['template_len'].values())) for S in sessions if S['session'] in mm):.0f} to {max(np.median(list(S['template_len'].values())) for S in sessions if S['session'] in mm):.0f} units long (mm):", sorted(set(S['subject'] for S in sessions if S['session'] in mm)))
print("template path lengths in m (metric sessions): %.2f to %.2f" % (
    min(min(S["template_len"].values()) * tg.length_scale(S) for S in sessions if np.isfinite(tg.length_scale(S))),
    max(max(S["template_len"].values()) * tg.length_scale(S) for S in sessions if np.isfinite(tg.length_scale(S)))))
print("template points per condition:", sorted(set(S["template_points"] for S in sessions)))
print("phase reference other than a CA1 channel:", {S["session"]: S["phase_ref"] for S in sessions if S["phase_ref"] not in ("ca1", None)})
print("LFP channels flagged (sessions with ca1, ca3, ls):",
      sum(bool(S["chan"]["ca1"]) for S in sessions), sum(bool(S["chan"]["ca3"]) for S in sessions), sum(bool(S["chan"]["ls"]) for S in sessions))
# clocks: linearized samples inside their trial; spikes against LFP; spikes against position
inside = np.mean([np.mean((S["trials"]["lints"][j] >= S["trials"]["start"][j] - 1e-6) & (S["trials"]["lints"][j] <= S["trials"]["stop"][j] + 1e-6))
                  for S in sessions for j in range(len(S["trials"]["start"])) if len(S["trials"]["lints"][j])])
print(f"fraction of linearized-position samples inside their trial interval: {inside:.4f}")
rl = []
for S in sessions:
    R0, R1 = [], []
    for U in S["units"] or []:
        if U["region"] in tg.HPC_LABELS and len(U["t"]) > 200 and "phase" in U:
            R0.append(np.abs(np.mean(np.exp(1j * U["phase"]))))
            R1.append(np.abs(np.mean(np.exp(1j * U["phase_shift"]))))
    if R0:
        rl.append((S["session"], np.median(R0), np.median(R1)))
rl = pd.DataFrame(rl, columns=["session", "R_true", "R_shift1s"])
print(f"hippocampal theta locking (median resultant length per session): true spike times {rl.R_true.median():.3f}, "
      f"spikes shifted 1 s {rl.R_shift1s.median():.3f}; sessions where true > shifted: {(rl.R_true > rl.R_shift1s).sum()}/{len(rl)}")
print("sessions where it is not:", rl[rl.R_true <= rl.R_shift1s].session.tolist())
cr = []
rng_clock = np.random.default_rng(7)
for S in sessions:
    m0, m1 = [], []
    for U in S["units"] or []:
        if U["region"] not in tg.HPC_LABELS:
            continue
        for c in tg.included_conditions(S, U)[:2]:
            M = tg.maps(S, U, c, "a")
            Us = dict(U, bin=U["bin_shift"])
            Ms = tg.maps(S, Us, c, "a")
            m0.append(np.nanmean(tg.decode(tg.box_fast(M["rate"], 20), rng_clock, "a", n_splits=3)))
            m1.append(np.nanmean(tg.decode(tg.box_fast(Ms["rate"], 20), rng_clock, "a", n_splits=3)))
    if m0:
        cr.append((S["session"], np.median(m0), np.median(m1)))
cr = pd.DataFrame(cr, columns=["session", "mse_true", "mse_shift1s"])
print(f"hippocampal rate decoding (median MSE per session, bins^2): spike positions as recorded {cr.mse_true.median():.0f}, "
      f"taken 1 s later {cr.mse_shift1s.median():.0f}; sessions where recorded < shifted: {(cr.mse_true < cr.mse_shift1s).sum()}/{len(cr)}")
print("sessions where it is not:", cr[cr.mse_true >= cr.mse_shift1s].session.tolist())
print("unit-conditions analysed by region:", D.region.value_counts().to_dict())
print("undefined smoothed phase (fraction of trial-by-bin samples), median: LS %.2f, hippocampal %.2f" % (
    D[D.region == "ls"]["aa0_phase_defined"].rsub(1).median(), D[D.region != "ls"]["aa0_phase_defined"].rsub(1).median()))
print("bins per condition, reading (b) 1.5 cm: %d to %d (reading (a): 201)" % (np.nanmin(D.nbins_b), np.nanmax(D.nbins_b)))

## ls-neuron-count
> A total of 1,647 LS neurons were recorded from five animals, in 62 sessions.

In [ ]:
ls_units = UN[UN.region == "ls"]
print("LS units:", len(ls_units), "| sessions with LS units:", ls_units.session.nunique(), "| animals:", ls_units.subject.nunique())
print(ls_units.groupby("subject").agg(units=("unit", "size"), sessions=("session", "nunique")))

**Result: reproduced.** 1,696 units are labelled LS (1,647 +/- 164.7), in 62 sessions (62 +/- 6.2), from all five animals.

## hpc-unit-count
> The hippocampal recordings hold 2,297 isolated CA1 units and 654 CA3 units (2,951 in all), with 1,587 CA1 and 224 CA3 place fields.

In [ ]:
ca3_sessions = {S["session"] for S in sessions if S["chan"]["ca3"]}
hp = UN[UN.region.isin(tg.HPC_LABELS)].copy()
hp["sub_session"] = np.where(hp.session.isin(ca3_sessions), "CA3", "CA1")
hp["sub_label"] = np.where(hp.region == "ca3", "CA3", "CA1")
print("hippocampal units:", len(hp), "| labels:", hp.region.value_counts().to_dict())
print("session reading (CA3 when the session has a CA3 site):", hp.sub_session.value_counts().to_dict())
print("unit label reading:", hp.sub_label.value_counts().to_dict())
Dh = D[D.region.isin(tg.HPC_LABELS)].copy()
Dh["sub_session"] = np.where(Dh.session.isin(ca3_sessions), "CA3", "CA1")
Dh["sub_label"] = np.where(Dh.region == "ca3", "CA3", "CA1")
for pr in "ab":
    print(f"place fields (conditions with a field, rate map reading {pr}):",
          Dh[Dh[f"pf{pr}_n"] > 0].groupby("sub_session").size().to_dict(), "(session reading);",
          Dh[Dh[f"pf{pr}_n"] > 0].groupby("sub_label").size().to_dict(), "(unit labels)")

**Result: not reproduced.** The files hold 3,260 hippocampal units (2,951 in the paper). By session (CA3 where the session has a CA3 site) they split into 2,261 CA1 and 999 CA3; by unit label into 2,636 'hpc' and 624 'ca3'. Under the session reading the CA1 count is within 229.7 of 2,297 and the CA3 count is not; under unit labels the CA3 count is within 65.4 of 654 and the CA1 count is not. No reading puts both within tolerance. The place-field counts are reported above, not judged.

## ls-active-neurons
> Of the 1,647 LS neurons, 840 fired at least 1.5 action potentials per trial on average.

In [ ]:
act_a = ls_units[ls_units.n_incl > 0]
act_b = ls_units[ls_units.n_in_trials >= 1.5 * ls_units.ntrials]
print(f"(a) LS units with at least one included condition: {len(act_a)} of {len(ls_units)}")
print(f"(b) LS units with spikes over all trials >= 1.5 x trials: {len(act_b)} of {len(ls_units)}")
print(act_a.groupby("subject").size().to_dict())
ACTIVE = set(zip(act_a.session, act_a.unit))

**Result: reproduced** under reading (a). 829 LS units have at least one included condition (840 +/- 84); under reading (b), spikes over all of the session's trials, 708 pass.

In [ ]:
# Per-condition tests over the 10 splits, for every reading
def p_less(a, b):
    a, b = np.asarray(a), np.asarray(b)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return stats.ttest_ind(a, b).pvalue
RD = [r for r in tg.READINGS if f"{r}_rate" in D]
for rd in RD:
    has = D[f"{rd}_rate"].notna()
    D.loc[has, f"{rd}_mrate"] = D.loc[has, f"{rd}_rate"].apply(np.nanmean)
    D.loc[has, f"{rd}_mphase"] = D.loc[has, f"{rd}_phase_cos"].apply(np.nanmean)
    D.loc[has, f"{rd}_mchance"] = D.loc[has, f"{rd}_chance_phase"].apply(np.nanmean)
    D.loc[has, f"{rd}_d"] = D.loc[has, f"{rd}_mphase"] - D.loc[has, f"{rd}_mrate"]
    D.loc[has, f"{rd}_p_vs_chance"] = [p_less(a, b) for a, b in zip(D.loc[has, f"{rd}_phase_cos"], D.loc[has, f"{rd}_chance_phase"])]
    D.loc[has, f"{rd}_p_vs_rate"] = [p_less(a, b) for a, b in zip(D.loc[has, f"{rd}_phase_cos"], D.loc[has, f"{rd}_rate"])]
    D[f"{rd}_sig_chance"] = (D[f"{rd}_p_vs_chance"] < 0.01) & (D[f"{rd}_mphase"] < D[f"{rd}_mchance"])
    D[f"{rd}_sig_rate"] = (D[f"{rd}_p_vs_rate"] < 0.01) & (D[f"{rd}_mphase"] < D[f"{rd}_mrate"])
NAMES = {"aa0": "split trials, 201 bins, undefined phase left out", "ba0": "split samples, 201 bins, left out",
         "aa1": "split trials, 201 bins, undefined phase = 0", "ba1": "split samples, 201 bins, = 0",
         "ab0": "split trials, 1.5 cm bins, left out", "bb0": "split samples, 1.5 cm bins, left out"}
DL = D[D.region == "ls"].copy()
DL["key"] = list(zip(DL.session, DL.unit))
print("LS included conditions:", len(DL), "| neurons:", DL.key.nunique())
print("readings:", NAMES)
print("conditions under the 1.5 cm reading that keep 201 bins (pixel sessions, DT2):", int(DL["ab0_rate"].isna().sum()))

## fig2-ls-phase-vs-shuffle
> The majority of LS neurons (751 of 840, 89%) showed position encoding in their spike theta phase, compared with shuffled data, in at least one condition.

In [ ]:
out = []
for rd in RD:
    sub = DL[DL[f"{rd}_rate"].notna() | (rd[1] == "a")]
    g = sub.groupby("key")[f"{rd}_sig_chance"].any()
    g = g.reindex(sorted(ACTIVE) if rd[1] == "a" else [k for k in sorted(ACTIVE) if k in set(sub.key)], fill_value=False)
    out.append({"reading": rd, "what": NAMES[rd], "neurons": len(g), "significant": int(g.sum()), "fraction": g.mean(),
                "conditions_sig": f"{int(sub[f'{rd}_sig_chance'].sum())}/{len(sub)}"})
fig2a = pd.DataFrame(out)
print(fig2a.to_string(index=False))
print("per animal, main reading:", DL.groupby(["subject", "key"])["aa0_sig_chance"].any().groupby("subject").mean().round(3).to_dict())

**Result: reproduced**, resting on the assumption that bins without a defined phase are left out. With samples split at random (reading ba0) 738 of 829 active LS neurons (0.890; 0.89 +/- 0.089) have phase MSE below chance at p < 0.01 in at least one condition. With whole trials split (aa0) the fraction is 0.643, and with 1.5 cm bins 0.608 (trials) and 0.830 (samples). When undefined phase is set to 0 the fraction is 0.462 to 0.578.

## fig2-ls-phase-beats-rate
> Spike-phase decoding outperformed rate decoding of the rat's position for 786 of 840 LS neurons, 3,046 of 4,393 conditions, and 62 of 62 sessions.

In [ ]:
out = []
for rd in RD:
    sub = DL[DL[f"{rd}_rate"].notna()]
    g = sub.groupby("key")[f"{rd}_sig_rate"].any()
    g = g.reindex(sorted(ACTIVE) if rd[1] == "a" else [k for k in sorted(ACTIVE) if k in set(sub.key)], fill_value=False)
    ca = sub[f"{rd}_sig_rate"].mean()
    cb = (sub[f"{rd}_mphase"] < sub[f"{rd}_mrate"]).mean()
    ss = sub.groupby("session")[[f"{rd}_mphase", f"{rd}_mrate"]].mean()
    out.append({"reading": rd, "neurons": f"{int(g.sum())}/{len(g)}", "neuron_frac": round(g.mean(), 3),
                "cond_frac_test": round(ca, 3), "cond_frac_lower": round(cb, 3), "conditions": len(sub),
                "sessions": f"{int((ss.iloc[:,0] < ss.iloc[:,1]).sum())}/{len(ss)}",
                "session_frac": round((ss.iloc[:, 0] < ss.iloc[:, 1]).mean(), 3)})
fig2b = pd.DataFrame(out)
print(fig2b.to_string(index=False))
print("expected: neurons 0.936 +/- 0.064, conditions 0.693 +/- 0.069, sessions 1.0 +/- 0.1")

**Result: partially reproduced**, resting on the same assumption. Phase is better than rate for most neurons, conditions and sessions under every reading where undefined phase is left out, but no reading puts all three fractions within tolerance. The closest, ba0, gives 0.835 of neurons (0.936 +/- 0.064), 0.731 of conditions by lower mean (0.693 +/- 0.069; 0.544 by the t test) and 60 of 60 sessions. When undefined phase is set to 0, phase beats rate for only 0.182 to 0.246 of neurons and 0.317 to 0.400 of sessions.

## fig2e-ls-phase-lower-mse
> Across LS neurons the spike-phase code is stronger than the rate code (two-sample t tests; N = 4,393 conditions, p < 1e-100; N = 62 sessions, p < 1e-10).

In [ ]:
out = []
for rd in RD:
    sub = DL[DL[f"{rd}_rate"].notna()]
    tc = stats.ttest_ind(sub[f"{rd}_mphase"], sub[f"{rd}_mrate"])
    ss = sub.groupby("session")[[f"{rd}_mphase", f"{rd}_mrate"]].mean()
    ts_ = stats.ttest_ind(ss.iloc[:, 0], ss.iloc[:, 1])
    out.append({"reading": rd, "n_cond": len(sub), "phase_mean": round(sub[f"{rd}_mphase"].mean(), 1),
                "rate_mean": round(sub[f"{rd}_mrate"].mean(), 1), "p_cond": tc.pvalue, "n_sess": len(ss),
                "phase_sess": round(ss.iloc[:, 0].mean(), 1), "rate_sess": round(ss.iloc[:, 1].mean(), 1), "p_sess": ts_.pvalue})
fig2e = pd.DataFrame(out)
print(fig2e.to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].hist(DL["aa0_mrate"], 50, alpha=.6, label="rate"); ax[0].hist(DL["aa0_mphase"], 50, alpha=.6, label="phase")
ax[0].set_xlabel("MSE (bins^2), reading aa0"); ax[0].legend()
ax[1].hist(DL["aa0_d"], 60); ax[1].set_xlabel("phase - rate MSE, LS conditions")
plt.tight_layout(); plt.show()
print("plotted ranges: rate %.0f to %.0f, phase %.0f to %.0f, difference %.0f to %.0f" % (
    DL.aa0_mrate.min(), DL.aa0_mrate.max(), DL.aa0_mphase.min(), DL.aa0_mphase.max(), DL.aa0_d.min(), DL.aa0_d.max()))

**Result: reproduced**, resting on the same assumption. Under aa0 the mean phase MSE over 4,362 LS conditions is 6,385 against 7,601 for rate (p = 7.8e-214), and over 60 sessions 6,349 against 7,700 (p = 2.0e-23). Reading ba0 passes both bounds as well; with 1.5 cm bins, ab0 misses both (p = 2.0e-30 and 0.028) and bb0 misses the session bound (p = 4.6e-07). When undefined phase is set to 0, phase MSE is higher than rate MSE (7,739 against 7,597).

## fig2fg-mse-difference
> The mean phase minus rate MSE is about -1,000 for LS conditions and -33 for hippocampal place fields, and -1,024 and -25 for LS and hippocampal sessions.

In [ ]:
out = []
for rd in RD:
    for pr in "ab":
        ls_ = DL[DL[f"{rd}_rate"].notna()]
        hpf = D[D.region.isin(tg.HPC_LABELS) & (D[f"pf{pr}_n"] > 0) & D[f"{rd}_rate"].notna()]
        out.append({"reading": rd, "fields": pr, "LS_cond": round(ls_[f"{rd}_d"].mean(), 1), "n_LS": len(ls_),
                    "HPC_fields": round(hpf[f"{rd}_d"].mean(), 1), "n_fields": len(hpf),
                    "LS_sess": round(ls_.groupby("session")[f"{rd}_d"].mean().mean(), 1), "n_LS_sess": ls_.session.nunique(),
                    "HPC_sess": round(hpf.groupby("session")[f"{rd}_d"].mean().mean(), 1), "n_HPC_sess": hpf.session.nunique()})
fig2fg = pd.DataFrame(out)
print(fig2fg.to_string(index=False))
print("expected: -1000 +/- 100, -33 +/- 3.3, -1024 +/- 102.4, -25 +/- 2.5")
print("LS by animal, main reading:", DL.groupby("subject").aa0_d.mean().round(0).to_dict())

**Result: not reproduced.** Under aa0 the mean phase minus rate MSE is -1,216 for 4,362 LS conditions and -3,871 for 1,640 hippocampal place fields; -1,351 for 60 LS sessions and -3,093 for 44 hippocampal sessions. The hippocampal values are about 100 times the paper's -33 and -25, and larger in magnitude than the LS values, which the partial rule excludes. No other reading brings the hippocampal values near -33. When undefined phase is set to 0 all four values are positive (101 to 277).

## fig1i-hpc-rate-phase-equivalent
> For hippocampal place fields the distributions of rate and phase decoding errors are statistically equivalent (two-sample KS test; 1,820 place fields, p = 0.19; 45 sessions, p = 0.61).

In [ ]:
out = []
for rd in RD:
    for pr in "ab":
        hpf = D[D.region.isin(tg.HPC_LABELS) & (D[f"pf{pr}_n"] > 0) & D[f"{rd}_rate"].notna()]
        k1 = stats.ks_2samp(hpf[f"{rd}_mrate"], hpf[f"{rd}_mphase"])
        ss = hpf.groupby("session")[[f"{rd}_mrate", f"{rd}_mphase"]].mean()
        k2 = stats.ks_2samp(ss.iloc[:, 0], ss.iloc[:, 1])
        out.append({"reading": rd, "fields": pr, "n_fields": len(hpf), "p_fields": k1.pvalue, "D_fields": round(k1.statistic, 3),
                    "mean_diff": round(hpf[f"{rd}_d"].mean(), 1), "n_sess": len(ss), "p_sess": k2.pvalue})
fig1i = pd.DataFrame(out)
print(fig1i.to_string(index=False))

**Result: not reproduced**, on the adopted assumption (undefined phase left out). The KS test finds rate and phase MSE different over 1,640 place fields (p = 3.7e-146; phase lower by 3,871) and over 44 sessions (p = 9.1e-11). When undefined phase is set to 0, the equivalence holds for the fields of rate map (b) (p = 0.21 and 0.24 over 1,924 fields, 0.82 over 45 sessions). It fails for the fields of rate map (a) (p = 0.044 and 0.0081).

## Place fields
Shared by ls-few-place-fields, hpc-place-field-properties, ls-place-fields-unstable and fig6e.

## ls-few-place-fields
> Only 5.3% (232 of 4,393) of LS rate maps showed place fields by the criteria applied to hippocampal units, where 31.7% (1,820 of 5,735) did.

In [ ]:
DH = D[D.region.isin(tg.HPC_LABELS)]
for pr in "ab":
    print(f"rate map reading {pr}: LS {int((DL[f'pf{pr}_n']>0).sum())}/{len(DL)} = {(DL[f'pf{pr}_n']>0).mean():.3f}; "
          f"hippocampal {int((DH[f'pf{pr}_n']>0).sum())}/{len(DH)} = {(DH[f'pf{pr}_n']>0).mean():.3f}")
    print("   by animal, LS:", DL.groupby("subject")[f"pf{pr}_n"].apply(lambda x: round((x > 0).mean(), 3)).to_dict(),
          "hippocampal:", DH.groupby("subject")[f"pf{pr}_n"].apply(lambda x: round((x > 0).mean(), 3)).to_dict())

**Result: partially reproduced.** With the Gaussian rate maps (a), 277 of 4,362 LS maps (0.064) and 1,640 of 5,723 hippocampal maps (0.287) have a field; with box maps (b), 0.100 and 0.336. The hippocampal fraction and both map counts are within tolerance; the LS fraction is lower than the hippocampal one but above 0.053 +/- 0.0053 under both readings.

## hpc-place-field-properties
> 711 of 1,425 hippocampal neurons had at least one place field (1.27 fields per neuron on average); the median peak in-field rate was 8.5 Hz and the median field width 38 bins (about 57 cm).

In [ ]:
hpu = UN[UN.region.isin(tg.HPC_LABELS)]
for pr in "ab":
    nf = DH.groupby(["session", "unit"])[f"pf{pr}_n"].sum()
    peaks = np.concatenate([x for x in DH[f"pf{pr}_peaks_all"] if isinstance(x, list)])
    widths = np.concatenate([x for x in DH[f"pf{pr}_widths_all"] if isinstance(x, list)])
    for den, n in (("a: units with an included condition", len(nf)), ("b: all hippocampal units", len(hpu))):
        print(f"reading {pr}, denominator {den}: {int((nf > 0).sum())}/{n} = {(nf > 0).sum()/n:.3f} with a field; "
              f"fields per neuron {nf.sum()/n:.2f}")
    print(f"   all fields: {len(peaks)}, median peak {np.median(peaks):.2f} Hz, median width {np.median(widths):.1f} bins")

**Result: partially reproduced.** With the Gaussian maps and the units that have an included condition (1,420, against the paper's 1,425), 677 have a field (0.477; 0.499 +/- 0.0499) and there are 1.27 fields per neuron (1.27 +/- 0.127); the median peak rate is 7.30 Hz (8.5 +/- 0.85) and the median width 43 bins (38 +/- 3.8). Two of the four values are within tolerance; the box maps give 0.532, 2.70, 6.00 Hz and 49 bins (one of four).

## ls-place-fields-unstable
> The few LS rate maps with a place field were less stable across trials than CA1/CA3 place fields (median coefficient of variance 41.5 against 10; two-sample t test, p < 1e-20).

In [ ]:
for pr in "ab":
    L_ = DL[DL[f"pf{pr}_n"] > 0]; H_ = DH[DH[f"pf{pr}_n"] > 0]
    for q, name in (("sd", "SD at the peak, Hz"), ("cv", "SD / mean at the peak, %")):
        t_ = stats.ttest_ind(L_[f"pf{pr}_{q}"], H_[f"pf{pr}_{q}"])
        print(f"rate maps {pr}, {name}: LS median {L_[f'pf{pr}_{q}'].median():.2f} (n={len(L_)}), "
              f"hippocampal {H_[f'pf{pr}_{q}'].median():.2f} (n={len(H_)}); t = {t_.statistic:.2f}, p = {t_.pvalue:.2e}")

**Result: partially reproduced.** As SD over trials divided by the mean at the peak (reading b), LS fields are more variable than hippocampal fields: medians 81.7 against 71.7% (Gaussian maps, p = 9.6e-13) and 85.1 against 71.7% (box maps, p = 4.0e-24). The medians are far from 41.5 and 10. As the SD in Hz (reading a), LS fields vary less than hippocampal ones (3.16 against 5.15 Hz).

## fig1e-phase-info-smaller-scale
> Hippocampal place fields reach their peak information at smaller smoothing scales for phase than for rate (medians 15 and 29 bins; two-sample t test, p < 1e-40).

In [ ]:
def best_scale(curve):
    c = np.convolve(np.nan_to_num(curve), np.ones(5) / 5, "same")
    return int(np.argmax(c)) + 1
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna()]
    for mode in "ab":
        br = np.array([best_scale(c) for c in H_[f"info_rate_{mode}"]])
        bp = np.array([best_scale(c) for c in H_[f"info_phase_{mode}"]])
        keep = (br < 49) & (bp < 49)
        t_ = stats.ttest_ind(bp[keep], br[keep])
        print(f"fields {pr}, levels {mode}: {keep.sum()} of {len(H_)} fields kept; median best smoothing phase {np.median(bp[keep]):.1f}, "
              f"rate {np.median(br[keep]):.1f} bins; t = {t_.statistic:.2f}, p = {t_.pvalue:.2e}")

**Result: partially reproduced.** Phase information peaks at smaller smoothing than rate information under every reading (p = 1.3e-34 to 1.2e-62). With equal-width levels the medians are 7 and 20.5 to 21 bins; with equal-count levels 20 to 21 and 27 to 28 bins. The rate median is within 29 +/- 2.9 under equal-count levels, but no reading puts the phase median within 15 +/- 1.5.

## fig1g-phase-more-info-fine-scale
> At smoothing below 6 cm (about one theta cycle) phase carries more spatial information than rate in hippocampal place fields, and above 10 cm rate carries more.

In [ ]:
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna()]
    for mode in "ab":
        R = np.stack(H_[f"info_rate_{mode}"]); P = np.stack(H_[f"info_phase_{mode}"])
        res = {}
        for w in range(2, 51):
            t_ = stats.ttest_ind(P[:, w - 1], R[:, w - 1]); k_ = stats.ks_2samp(P[:, w - 1], R[:, w - 1])
            res[w] = (np.mean(P[:, w - 1] - R[:, w - 1]), t_.pvalue, k_.pvalue)
        fine = [res[w] for w in (2, 3)]
        coarse = [res[w] for w in range(7, 51)]
        for name, i, thr in (("t test p<1e-5", 1, 1e-5), ("KS p<0.01", 2, 0.01)):
            f_ok = all(r[0] > 0 and r[i] < thr for r in fine)
            c_frac = np.mean([r[0] < 0 and r[i] < thr for r in coarse])
            print(f"fields {pr}, levels {mode}, {name}: phase-rate at 2 and 3 bins {fine[0][0]:.3f} (p {fine[0][i]:.1e}), "
                  f"{fine[1][0]:.3f} (p {fine[1][i]:.1e}); phase > rate at both: {f_ok}; "
                  f"rate > phase significant at {c_frac:.2f} of smoothings 7-50")
        d_all = np.array([res[w][0] for w in range(2, 51)])
        print(f"   phase-rate difference over smoothings 2-50: {d_all.min():.3f} to {d_all.max():.3f}; first smoothing with rate > phase: "
              f"{next((w for w in range(2, 51) if res[w][0] < 0), None)}")

**Result: partially reproduced.** At smoothings of 2 and 3 bins phase carries more information than rate under every reading (differences 2.06 to 2.80 bits, p below both thresholds). Rate never carries more than phase: the difference stays positive at every smoothing from 2 to 50, so the coarse-scale half fails.

## ls-lower-peak-info
> LS rate maps had lower peak information rates than hippocampal place field maps (average 0.4 against 1.7 bits/cm; two-sample t test, p < 1e-100).

In [ ]:
L_ = DL[DL["info_rate_a"].notna()]
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna()]
    for mode in "ab":
        for sm in ("a: peak over 1-50", "b: 20 bins"):
            def val(df):
                C = np.stack(df[f"info_rate_{mode}"])
                return C.max(1) if sm[0] == "a" else C[:, 19]
            vl, vh = val(L_), val(H_)
            for unit in ("a: /1.5 cm", "b: /bin width"):
                dl = 1.5 if unit[0] == "a" else L_.cm_per_bin_a.values
                dh = 1.5 if unit[0] == "a" else H_.cm_per_bin_a.values
                a_, b_ = vl / dl, vh / dh
                t_ = stats.ttest_ind(a_, b_)
                print(f"fields {pr}, levels {mode}, smoothing {sm}, unit {unit}: LS {a_.mean():.3f} (n={len(a_)}), "
                      f"hippocampal {b_.mean():.3f} (n={len(b_)}) bits/cm; t = {t_.statistic:.1f}, p = {t_.pvalue:.1e}")

**Result: partially reproduced.** LS rate maps have lower peak information than hippocampal place fields under all 16 readings (p from 8.7e-150 to below 1e-300). No reading puts both averages within tolerance. Peak over smoothings at 1.5 cm per bin gives hippocampal 1.659 bits/cm (1.7 +/- 0.17) with LS 0.965; 20-bin smoothing with equal-count levels gives LS 0.416 (0.4 +/- 0.04) with hippocampal 1.045.

## fig3-ls-phase-reliable
> Without preselection of place fields, the LS phase code for position is more reliable across the population than either the rate or the phase code of the hippocampus.

In [ ]:
def corr_rate(M):
    return np.corrcoef(M.T)
def corr_circ(P):
    P = np.where(np.isnan(P), np.nan, P)
    mu = np.arctan2(np.nanmean(np.sin(P), 0), np.nanmean(np.cos(P), 0))
    Sx = np.nan_to_num(np.sin(P - mu))
    num = Sx.T @ Sx
    d = np.sqrt(np.diag(num))
    return num / np.outer(d, d)
def measures(C):
    n = C.shape[0]; i = np.arange(n)
    anti = C[n - 1 - i, i]
    a = np.nanmean(anti[np.abs(i - (n - 1) / 2) > 20])
    off = np.abs(i[:, None] - i[None, :]) > 20
    return a, np.nanmean(C[off])
def to201(m, circular=False):
    # maps of conditions with 151 template points are interpolated onto 201 (S)
    m = np.asarray(m, float)
    if len(m) == 201:
        return m
    x0, x1 = np.linspace(0, 1, len(m)), np.linspace(0, 1, 201)
    if circular:
        return np.arctan2(np.interp(x1, x0, np.sin(m)), np.interp(x1, x0, np.cos(m)))
    return np.interp(x1, x0, m)
n151 = int(sum(len(m) != 201 for m in D.mean_rate_map))
print(f"conditions with maps of other than 201 points, interpolated: {n151}")
RM = {nm: np.stack([to201(m) for m in df.mean_rate_map]) for nm, df in (("HPC", DH), ("LS", DL))}
PM = {nm: np.stack([to201(m, True) for m in df.mean_phase_map]) for nm, df in (("HPC", DH), ("LS", DL))}
res = {}
for name, df in (("HPC", DH), ("LS", DL)):
    R = RM[name]; P = PM[name]
    res[(name, "rate")] = measures(corr_rate(R)); res[(name, "phase")] = measures(corr_circ(P))
    print(f"{name}: {len(df)} conditions; rate: anti-diagonal {res[(name,'rate')][0]:.3f}, off-diagonal {res[(name,'rate')][1]:.3f}; "
          f"phase: anti-diagonal {res[(name,'phase')][0]:.3f}, off-diagonal {res[(name,'phase')][1]:.3f}")
for k, nm in ((0, "(a) anti-diagonal"), (1, "(b) off-diagonal")):
    v = res[("LS", "phase")][k]
    print(f"{nm}: LS phase {v:.3f} lower than HPC rate {res[('HPC','rate')][k]:.3f}: {v < res[('HPC','rate')][k]}; "
          f"lower than HPC phase {res[('HPC','phase')][k]:.3f}: {v < res[('HPC','phase')][k]}")
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for a_, (nm, C) in zip(ax, [("HPC rate", corr_rate(RM["HPC"])), ("HPC phase", corr_circ(PM["HPC"])),
                            ("LS rate", corr_rate(RM["LS"])), ("LS phase", corr_circ(PM["LS"]))]):
    im = a_.imshow(C, vmin=-1, vmax=1, cmap="RdBu_r"); a_.set_title(nm)
plt.tight_layout(); plt.show()
print("colour axis -1 to 1; correlations lie in [-1, 1], none outside")

**Result: reproduced** under both readings. Across 4,362 LS conditions the LS phase correlation is 0.036 on the anti-diagonal away from the centre and 0.070 off the diagonal. That is lower than hippocampal rate (0.868 and 0.891) and hippocampal phase (0.227 and 0.231) across 5,723 conditions.

## fig4d-depth-gradient
> The strength of phase coding relative to rate coding increases with recording depth within the LS, within sessions and across animals.

In [ ]:
sess_by = {S["session"]: S for S in sessions}
rows = []
for s_, g in DL.groupby("session"):
    S = sess_by[s_]
    first_ls = next(U for U in S["units"] if U["region"] == "ls")
    rows.append({"session": s_, "subject": S["subject"], "depth": S["depth"] + 10 * first_ls["site"], "d": g.aa0_d.mean()})
DD = pd.DataFrame(rows)
def across(DD):
    out = {}
    for sub, g in DD.groupby("subject"):
        if len(g) >= 4:
            out[sub] = stats.pearsonr(g.depth, g.d)
    c = DD.copy()
    c["dc"] = c.groupby("subject").depth.transform(lambda x: x - x.mean()); c["vc"] = c.groupby("subject").d.transform(lambda x: x - x.mean())
    return out, stats.pearsonr(c.dc, c.vc)
for label, dd in (("all sessions", DD), ("without the last DT9 session", DD.drop(DD[DD.subject == "DT9"].sort_values("session").index[-1:]))):
    per, pooled = across(dd)
    print(label, "| per animal r (p):", {k: (round(v[0], 3), round(v[1], 3)) for k, v in per.items()},
          f"| (a) all r < 0: {all(v[0] < 0 for v in per.values())} | (b) pooled r = {pooled[0]:.3f}, p = {pooled[1]:.3g}")
print("sessions per animal:", DD.subject.value_counts().to_dict())
wr = []
for s_, g in DL.groupby("session"):
    gu = g.groupby("unit").agg(site=("site", "first"), d=("aa0_d", "mean"))
    if gu.site.nunique() >= 5:
        wr.append((s_, stats.pearsonr(gu.site, gu.d)[0], len(gu)))
wr = pd.DataFrame(wr, columns=["session", "r", "units"])
print(f"within sessions with units on >= 5 sites: {len(wr)} sessions, r < 0 in {(wr.r < 0).sum()} ({(wr.r < 0).mean():.2f}); median r {wr.r.median():.3f}")

**Result: partially reproduced.** Across sessions the correlation of depth with the session's phase minus rate MSE is positive in all four animals with at least four LS sessions (DT2 0.022, DT5 0.245, DT8 0.226, DT9 0.480). The pooled within-animal correlation is also positive (r = 0.334, p = 0.009), opposite to the paper's direction, so the across-animal part fails. Within sessions with units on at least five sites the correlation is negative in 24 of 47 (0.51, median r -0.005), a bare majority, so that part holds as the criterion is written.

## fig4e-onset-depth
> The theta phase at the onset of LS phase fields varies with recording depth (DT5 r = 0.38, DT9 r = 0.25, DT2 r = 0.27; circular-linear correlation, p < 0.001).

In [ ]:
sig = DL[DL.aa0_sig_chance & DL.key.isin(ACTIVE)].copy()
sig["depth_a"] = [sess_by[s]["depth"] for s in sig.session]
sig["depth_b"] = sig.depth_a + 10 * sig.site
best = sig.loc[sig.groupby("key").aa0_mphase.idxmin()]
out = []
for r1, df in (("a each condition", sig), ("b best condition", best)):
    for r3 in ("onset_spikes", "onset_map"):
        for r4 in ("depth_a", "depth_b"):
            row = {"conditions": r1, "phase": r3, "depth": r4}
            for sub in ("DT5", "DT9", "DT2"):
                g = df[(df.subject == sub) & df[r3].notna()]
                r, p = tg.circ_corrcl(g[r3].values, g[r4].values) if len(g) > 3 else (np.nan, np.nan)
                row[sub] = f"r={r:.3f} p={p:.1e} n={len(g)}"
            out.append(row)
print(pd.DataFrame(out).to_string(index=False))
print("expected r: DT5 0.38 +/- 0.038, DT9 0.25 +/- 0.025, DT2 0.27 +/- 0.027, each p < 0.001")

**Result: not reproduced.** The onset phase is correlated with depth in each animal, but in no reading does any animal reach p < 0.001. With every significant condition, spike phases and the unit's depth: DT5 r = 0.144 (p = 0.018), DT9 r = 0.153 (p = 0.063), DT2 r = 0.257 (p = 0.0015). The DT2 r is within 0.27 +/- 0.027.

## fig4e-onset-mediolateral
> The starting phase of LS phase fields varies systematically along the medial-lateral axis across animals.

In [ ]:
# Medial-lateral LS coordinates (mm) as Figure S1A's labels give them (supplement, page 6):
# DT2 ".6 ML" and "1.5 ML"; DT5 ".5ML"; DT7 ".5 ML"; DT8 ".5 ML"; DT9 "1.5 ML".
ML_FIRST = {"DT2": 0.6, "DT5": 0.5, "DT7": 0.5, "DT8": 0.5, "DT9": 1.5}
ML_MEAN = {"DT2": (0.6 + 1.5) / 2, "DT5": 0.5, "DT7": 0.5, "DT8": 0.5, "DT9": 1.5}
for r3 in ("onset_spikes", "onset_map"):
    g = sig[sig[r3].notna()]
    means = g.groupby("subject")[r3].apply(tg.circ_mean)
    kw = stats.kruskal(*[x[r3].values for _, x in g.groupby("subject")])
    print(f"{r3}: animal circular means (rad) {means.round(2).to_dict()}, n {g.subject.value_counts().to_dict()}; Kruskal-Wallis p = {kw.pvalue:.2e}")
    for nm, ML in (("(a) first listed", ML_FIRST), ("(b) mean listed", ML_MEAN)):
        r, p = tg.circ_corrcl(means.values, np.array([ML[s] for s in means.index]))
        print(f"   ML {nm}: circ_corrcl r = {r:.3f} (p = {p:.2f})")

**Result: reproduced** under one reading. With the onset taken from the smoothed phase map, the animals' circular mean onsets differ (Kruskal-Wallis p = 0.017) and correlate with the medial-lateral coordinate (r = 0.916 with the first listed coordinate, 0.935 with the mean; five animals, p = 0.12 and 0.11). With the onset from spike phases the correlation is 0.840 to 0.880 but the animals do not differ (p = 0.37). Two of the five animals carry most of the spread (DT2 2.75 rad, DT9 0.84 rad).

## speed-independent
> Only 2.8% (125 of 4,393) of LS conditions showed a significant correlation between running speed and the rate of phase precession, and 10.7% of LS rate maps were modulated by running speed, so LS phase coding seems independent of running speed.

In [ ]:
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    sp_parts = list(ex.map(tg.speed_session, sessions, [seeds[S["session"]] + 1 for S in sessions]))
SP = [r for p in sp_parts for r in p]
print(f"{len(SP)} LS conditions in {time.time()-t:.0f} s")
for k, nm in (("a", "regression slope"), ("b", "circular-linear r")):
    v = [r["prec"][k] for r in SP if r["prec"] is not None and np.isfinite(r["prec"][k][1])]
    print(f"precession as {nm}: {sum(p < 0.01 for _, p in v)}/{len(v)} = {np.mean([p < 0.01 for _, p in v]):.3f} significant; "
          f"{len(SP) - len(v)} conditions with too few trials of 5+ spikes")
for k, nm in (("a", "per spike"), ("b", "100 ms bins")):
    v = [r["ratespeed"][k] for r in SP if r["ratespeed"] is not None and k in r["ratespeed"]]
    print(f"rate and speed {nm}: {sum(p < 0.01 for _, p in v)}/{len(v)} = {np.mean([p < 0.01 for _, p in v]):.3f} significant; median r {np.median([r for r, _ in v]):.3f}")

**Result: partially reproduced.** Both fractions are small, but both are below tolerance. Speed correlates with the rate of precession in 15 of 2,879 LS conditions (0.005, slope) or 23 of 2,877 (0.008, circular-linear r), against 0.028 +/- 0.0028; 1,483 conditions had too few trials with 5 spikes. Rate is modulated by speed in 0.051 (per spike) or 0.076 (100 ms bins) of 4,362 conditions, against 0.107 +/- 0.0107. Both are below 20%.

## fig5-assemblies
> Spike trains of hippocampal neurons predicted LS firing in 18.5% (2,087 of 11,264) of hippocampal-LS pairs, with a median optimal timescale of 25 ms.

In [ ]:
pairs, ms_tot, unit_pairs = 0, [], 0
for S in sessions:
    if not S["units"]:
        continue
    nh = sum(U["region"] in tg.HPC_LABELS for U in S["units"]); nl = sum(U["region"] == "ls" for U in S["units"])
    unit_pairs += nh * nl
    T = S["trials"]
    for c in np.unique(T["condition"]):
        m = T["condition"] == c
        if m.sum() >= tg.MIN_TRIALS and nh and nl:
            pairs += nh * nl
            ms_tot += [int(np.sum(T["stop"][m] - T["start"][m]) * 1000)] * (nh * nl)
ms_tot = np.array(ms_tot)
cost_pair, per_fit = tg.peer_prediction_cost(int(np.median(ms_tot)))
print(f"hippocampal-LS pairs by condition (conditions with >= 10 trials): {pairs}; median samples per pair {np.median(ms_tot):.0f} (1 ms bins)")
print(f"one least-squares GLM fit on {np.median(ms_tot):.0f} samples: {per_fit*1000:.1f} ms; one pair (150 windows x 11 fits): {cost_pair:.1f} s")
print(f"simultaneously recorded hippocampal-LS unit pairs (not by condition): {unit_pairs}")
print(f"estimated total for {unit_pairs} pairs: {unit_pairs * cost_pair / 3600:.1f} CPU hours")
print(f"estimated total: {pairs * cost_pair / 3600:.1f} CPU hours, {pairs * cost_pair / 3600 / 4:.1f} h on 4 CPUs")

**Result: blocked (too_large).** One pair needs 150 windows x 11 GLM fits on about 82,000 1 ms samples, about 13 s with the cheapest (least-squares) fit. The 104,845 simultaneously recorded hippocampal-LS unit pairs would take roughly 400 CPU hours, and the pairs by condition over 2,000. Even the paper's 11,264 pairs would take about 40 CPU hours (11,264 x 13 s), beyond the time limit. Nothing was fitted.

## fig6-session-counts
> LS neurons were recorded with CA1 neurons in 63 sessions and with CA3 neurons in 19 sessions, and 14 sessions had LFP from CA1, CA3 and LS together.

In [ ]:
g = UN.groupby("session").region.apply(set)
has_ls = g.apply(lambda s: "ls" in s)
hpc_s = g.apply(lambda s: bool(s & set(tg.HPC_LABELS)))
ca1_lab = g.apply(lambda s: bool(s & {"hpc", "ca1"})); ca3_lab = g.apply(lambda s: "ca3" in s)
ca3_site = g.index.isin(list(ca3_sessions))
print(f"LS with hippocampal units: {int((has_ls & hpc_s).sum())}")
print(f"session reading: LS with CA1 {int((has_ls & hpc_s & ~ca3_site).sum())}, LS with CA3 {int((has_ls & hpc_s & ca3_site).sum())}")
print(f"unit labels: LS with CA1 ('hpc'/'ca1') {int((has_ls & ca1_lab).sum())}, LS with CA3 ('ca3') {int((has_ls & ca3_lab).sum())}")
ls_ca1lfp = [S["session"] for S in sessions if S["units"] and any(U["region"] == "ls" for U in S["units"]) and S["chan"]["ca1"]]
ls_ca3lfp = [S["session"] for S in sessions if S["units"] and any(U["region"] == "ls" for U in S["units"]) and S["chan"]["ca3"]]
print(f"variant, by LFP channel: sessions with LS units and a CA1 channel {len(ls_ca1lfp)}, with a CA3 channel {len(ls_ca3lfp)}")
lfp3 = [S["session"] for S in sessions if S["chan"]["ca1"] and S["chan"]["ca3"] and S["chan"]["ls"]]
print(f"sessions with CA1, CA3 and LS LFP channels flagged: {len(lfp3)}; by animal {dict(Counter(sess_by[s]['subject'] for s in lfp3))}")

**Result: partially reproduced.** LS units were recorded with CA3 in 19 sessions under the session reading (19 +/- 1.9), and 14 sessions have CA1, CA3 and LS channels flagged (14 +/- 1.4). LS and CA1 units are together in only 25 sessions (session reading) or 33 (unit labels), against 63. Counting a session's CA1 by its flagged CA1 channel gives 57, within tolerance.

## fig6c-ca3-coherence-increases
> Theta phase coherence between LS and the CA3 pyramidal layer increases from the beginning to the end of the run, while coherence between LS and CA1 stays relatively flat.

In [ ]:
xs = np.linspace(0, 1, 1000)
def lfp_table(key):
    rows = []
    for S in sessions:
        L = S.get("lfp") or {}
        for reg in ("ca1", "ca3"):
            for r in L.get(reg, []):
                z = r[f"{key}_z"]
                if z is None:
                    continue
                rows.append({"session": S["session"], "subject": S["subject"], "reg": reg, "slope": np.polyfit(xs, z, 1)[0],
                             "raw": r[f"{key}_mean"], "z": z})
    return pd.DataFrame(rows)
def judge(T_, label):
    for lvl in ("trial", "session"):
        X = T_ if lvl == "trial" else T_.groupby(["session", "reg"]).agg(slope=("slope", "mean"), raw=("raw", "mean")).reset_index()
        a1, a3 = X[X.reg == "ca1"], X[X.reg == "ca3"]
        o1, o3 = stats.ttest_1samp(a1.slope, 0), stats.ttest_1samp(a3.slope, 0)
        d = stats.ttest_ind(a3.slope, a1.slope)
        r_ = stats.ttest_ind(a3.raw, a1.raw)
        print(f"{label}, per {lvl}: CA1 slope {a1.slope.mean():.3f} (n={len(a1)}, p={o1.pvalue:.1e}); CA3 slope {a3.slope.mean():.3f} "
              f"(n={len(a3)}, p={o3.pvalue:.1e}); CA3 - CA1 slope p={d.pvalue:.1e}; raw mean CA1 {a1.raw.mean():.4f}, CA3 {a3.raw.mean():.4f}, p={r_.pvalue:.1e}")
    for reg in ("ca1", "ca3"):
        Z = np.stack(T_[T_.reg == reg].z)
        m = Z.mean(0)
        print(f"   {reg} mean z curve: last tenth - first tenth = {m[-100:].mean() - m[:100].mean():.3f}; range {m.min():.2f} to {m.max():.2f}")
COH = lfp_table("coh")
print("sessions with a CA1 channel:", COH[COH.reg == "ca1"].session.nunique(), "| with a CA3 channel:", COH[COH.reg == "ca3"].session.nunique())
judge(COH, "theta phase coherence")
print("CA3 slope by animal:", COH[COH.reg == "ca3"].groupby("subject").slope.mean().round(3).to_dict(), "| CA1:", COH[COH.reg == "ca1"].groupby("subject").slope.mean().round(3).to_dict())

**Result: reproduced.** Per trial, the slope of z-scored CA3-LS theta phase coherence over the run is 0.366 (2,699 trials, p = 3.6e-111) and exceeds the CA1-LS slope of 0.044 (p = 4.6e-88). Over sessions the CA3 slope is 0.259 (19 sessions, p = 0.0062), above CA1 (p = 0.0030). The CA1 mean curve changes by 0.030 from first to last tenth against 0.321 for CA3. One of the three CA3 animals (DT7, mean slope -0.171) runs the other way.

## fig6d-cfc-stronger-ca3
> Coupling between hippocampal theta phase and LS gamma amplitude increases along the run for both CA1 and CA3, and is stronger for CA3-LS than for CA1-LS.

In [ ]:
CFC = lfp_table("cfc"); judge(CFC, "theta-gamma coupling, 40-200 Hz, 50 ms RMS")
CFC2 = lfp_table("cfc2"); judge(CFC2, "variant 50-500 Hz, 60-sample RMS")
same = CFC[CFC.session.isin(CFC[CFC.reg == "ca3"].session)]
judge(same, "40-200 Hz, only sessions with both channels")

**Result: reproduced.** Theta-gamma coupling rises over the run for CA1 (slope 0.084, p = 1.5e-19) and CA3 (0.284, p = 1.1e-49), and raw coupling is higher for CA3 (0.376 against 0.345, p = 5.3e-187); per session the same holds (p = 4.8e-05, 2.5e-06 and 7.0e-08). The 50 to 500 Hz variant gives the same signs.

## figs7-ensemble-ls-phase-best
> The firing phases of ensembles of simultaneously recorded LS neurons read out position better than the firing phases or rates of an equal number of hippocampal neurons (p < 0.001 at 15 to 25 neurons).

In [ ]:
ENS_DRAWS = 3
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    ens = [r for p in ex.map(tg.ensemble_session, sessions, [seeds[S["session"]] + 2 for S in sessions], [ENS_DRAWS] * len(sessions)) for r in p]
E = pd.DataFrame(ens)
print(f"{len(E)} ensemble decodes in {time.time()-t:.0f} s; conditions with >= 15 included units: "
      f"LS {E[E.region=='LS'].groupby(['session','cond']).ngroups}, HPC {E[E.region=='HPC'].groupby(['session','cond']).ngroups}")
for reading in ("a", "b"):
    rows = []
    for size in range(15, 26):
        e = E[(E['size'] == size) & (E.reading == reading)]
        lp = e[(e.region == "LS") & (e.feature == "phase")].mse; hp_ = e[(e.region == "HPC") & (e.feature == "phase")].mse; hr = e[(e.region == "HPC") & (e.feature == "rate")].mse
        p1 = stats.ttest_ind(lp, hp_).pvalue if len(lp) > 1 and len(hp_) > 1 else np.nan
        p2 = stats.ttest_ind(lp, hr).pvalue if len(lp) > 1 and len(hr) > 1 else np.nan
        rows.append({"size": size, "n_LS": len(lp), "LS_phase": round(lp.mean(), 0), "n_HPC": len(hp_), "HPC_phase": round(hp_.mean(), 0),
                     "HPC_rate": round(hr.mean(), 0), "p_vs_HPC_phase": p1, "p_vs_HPC_rate": p2,
                     "LS_best_p001": bool(lp.mean() < hp_.mean() and lp.mean() < hr.mean() and p1 < 0.001 and p2 < 0.001)})
    print(f"reading ({reading}):"); print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** At every size from 15 to 25, LS phase ensembles decode position worse than hippocampal phase ensembles (for example 6,028 against 5,793 bins squared at 15 units, reading a) and much worse than hippocampal rate ensembles (2,004). The differences are significant in the wrong direction (p = 3.2e-07 and below 1e-300 at 15 units). Reading (b) gives the same.

## figs2-rate-phase-independent
> Instantaneous firing rate and firing phase are independent at the single-trial level; fewer than 2% of hippocampal conditions had a rate-phase correlation higher than a circular-shift null.

In [ ]:
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    IF = pd.DataFrame([r for p in ex.map(tg.ifr_session, sessions, [seeds[S["session"]] + 3 for S in sessions]) for r in p])
print(f"{len(IF)} conditions in {time.time()-t:.0f} s")
for reg, m in (("hippocampal", IF.region.isin(tg.HPC_LABELS)), ("LS", IF.region == "ls")):
    X = IF[m]
    a_ = ((X.t_p < 0.05) & (X.t_stat > 0)).mean(); b_ = (X.mean_act > X.pct95).mean()
    print(f"{reg}: {len(X)} conditions; mean actual r {X.mean_act.mean():.3f}, mean shuffled {X.mean_sh.mean():.3f}; "
          f"(a) t test higher at p<0.05: {a_:.4f}; (b) above the 95th percentile: {b_:.4f}")

**Result: not reproduced.** Over 5,453 hippocampal conditions the mean actual rate-phase correlation (0.435) is no higher than the circularly shifted one (0.444). But 0.143 of conditions are significantly higher by the t test (a) and 0.207 lie above the 95th percentile (b), against the paper's fewer than 2%. For LS the fractions are 0.032 and 0.112.

## figs1f-trial-counts
> The behaviour comprises 2,889 circle-alternation, 6,608 central-alternation and 625 linear-track trials; ballistic trials made up about 90 to 95% of trials in each recording.

In [ ]:
print("trials by task, all sessions:", dict(tasks))
tasks_ls = Counter(x for S in sessions if S["units"] and any(U["region"] == "ls" for U in S["units"]) for x in task_of(S))
print("trials by task, sessions with LS units:", dict(tasks_ls))
print("non-ballistic trials in the files: none (the trials table holds only the ballistic runs, by its description)")

**Result: partially reproduced.** In the 62 sessions with LS units there are 3,039 circle (wheel) alternation trials (2,889 +/- 288.9) and 625 linear-track trials (625 +/- 62.5), but 5,219 central-alternation trials against 6,608 +/- 660.8. Over all 67 sessions the counts are 3,987, 5,219 and 701, all outside tolerance. The ballistic fraction cannot be tested: the trials table keeps only the ballistic runs.

In [ ]:
print(f"notebook ran in {(time.time() - T0)/60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")

## Summary

| claim | outcome |
|---|---|
| ls-neuron-count | reproduced |
| hpc-unit-count | not reproduced |
| ls-active-neurons | reproduced |
| fig2-ls-phase-vs-shuffle | reproduced (assumed) |
| fig2-ls-phase-beats-rate | partially reproduced (assumed) |
| fig2e-ls-phase-lower-mse | reproduced (assumed) |
| fig2fg-mse-difference | not reproduced (assumed) |
| fig1i-hpc-rate-phase-equivalent | not reproduced (assumed) |
| ls-lower-peak-info | partially reproduced |
| ls-few-place-fields | partially reproduced |
| hpc-place-field-properties | partially reproduced |
| ls-place-fields-unstable | partially reproduced |
| fig1e-phase-info-smaller-scale | partially reproduced (assumed) |
| fig1g-phase-more-info-fine-scale | partially reproduced (assumed) |
| fig3-ls-phase-reliable | reproduced |
| fig4d-depth-gradient | partially reproduced (assumed) |
| fig4e-onset-depth | not reproduced (assumed) |
| fig4e-onset-mediolateral | reproduced (assumed) |
| speed-independent | partially reproduced |
| fig5-assemblies | blocked, too_large |
| fig5g-assembly-neurons-stronger-phase | blocked, too_large (depends on fig5-assemblies) |
| figs5-assembly-independent-of-locking | blocked, too_large (depends on fig5-assemblies) |
| fig6-session-counts | partially reproduced |
| fig6c-ca3-coherence-increases | reproduced |
| fig6d-cfc-stronger-ca3 | reproduced |
| fig6e-assembly-position | blocked, too_large (depends on fig5-assemblies) |
| figs7-ensemble-ls-phase-best | not reproduced (assumed) |
| figs2-rate-phase-independent | not reproduced |
| figs3-ls-major-target | blocked, other (Allen data not analysed in this run) |
| figs6-reference-frame | blocked, metadata_insufficient |
| figs1f-trial-counts | partially reproduced |
| fig7-dynamic-weighting-model | blocked, other (model not run in this run) |

"(assumed)" marks an outcome that rests on the assumption that position bins with no spike in their smoothing window are left out of phase decoding.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's.
- The paper does not say how bins without spikes enter phase decoding. The two treatments run here give opposite results. Left out, LS phase coding is strong but hippocampal phase decoding is far better than rate (against the paper's equivalence). Set to phase 0, the hippocampal equivalence holds but LS phase no longer beats rate.
- Place fields follow the Methods' rules, applied the way buzcode's bz_getPlaceFields1D applies them. The Gaussian width (5 bins) and the choice of the highest peak as the "first" field are assumptions.
- Olypher information uses normalized probabilities. The paper's division by trial count and its conversion to bits/cm are not reproduced exactly (two readings each).
- Bins of 1.5 cm cannot be formed for DT2, whose positions are in pixels; those sessions keep 201 bins under that reading.
- Spike phase is referenced to the CA1 channel the files flag; six sessions without one use the hippocampal probe's theta channel (five DT9 sessions with only a CA3 probe, one DT7 session).
- Speed is computed from x and y position smoothed over 0.25 s; precession slope by maximizing the mean resultant length.
- Peer prediction (Figure 5) was not run: estimated at hundreds of CPU hours. Figures 5G, 6E and S5 depend on it.
- The Allen connectivity analysis (Figure S3) and the spiking model (Figure 7) were not run in this attempt's time.

What was found about the dataset.
- 31 sessions of DT7, DT8 and DT9 store head position labelled "meters" whose values are millimetres: their template paths are 1,329 to 2,934 units long, against 0.97 to 3.13 m elsewhere.
- One DT7 session (20170324) has no units table.
- No CA1 channel is flagged in five DT9 sessions (CA3 probe only) and one DT7 session. The paper refers all spike phases to CA1 theta.
- Hippocampal units number 3,260 (2,636 'hpc', 624 'ca3'), against the paper's 2,951. The 'hpc' label is the probe, not the subfield, so CA1 against CA3 is a reading.
- No session is marked as holding maze rotations, so Figure S6 cannot be tested.
- Some files lack trials.trial_type, trial_conditions.condition_type or units.max_waveform_channel.
- Clocks: 99.7% of linearized samples lie inside their trial. Hippocampal spikes lock to theta in 47 of 47 sessions (median resultant length 0.179 against 0.053 with spikes shifted by 1 s). Rate decoding is better with recorded positions than with positions 1 s later in 44 of 47 sessions; the three exceptions are listed in the data check.
